In [1]:
from time import perf_counter  

from modeling import build_models_from_csv
from bundles import BaseBundle, MSBundle
from utils import (
    tighten_bounds_one_model,
    MIN_DIST, ACTIVE_TOL, GAP_STOP_TOL,
)
from simplex_specialstart import run_pid_simplex_3d

# setup scenario number and target_nodes
csv_path       = "data.csv"
max_scenarios  = 10
target_nodes   = 100


bounds={
    "Kp": (-1, 0),
    "Ki": (-101, -99),
    "Kd": (0, 1),
    "x": (-2.5, 2.5),
    "u": (-5.0, 5.0),
    "e": (None, None),
    "I": (None, None),
}
bounds={
    "Kp": (-10.0, 10.0),
    "Ki": (-100.0, 100.0),
    "Kd": (-100.0, 1000.0),
    "x": (-2.5, 2.5),
    "u": (-5.0, 5.0),
    "e": (None, None),
    "I": (None, None),
}

weights = (10.0, 0.01)


# ====== stage 1：load data and generate models ======
T   = 15.0          
nfe = 20            

t_load0 = perf_counter()
model_list, first_stg_vars_list, m_tmpl_list, nfe = build_models_from_csv(
    csv_path=csv_path,
    T=T,
    nfe=nfe,
    weights=weights,
    bounds=bounds,
    sp0=0.0,
    sp1=0.5,
    tau_xs_col="tau_xs",
    tau_us_col="tau_us",
    tau_ds_col="tau_ds",          
    disturb_prefix="disturbance_",
    setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios,
    skip=0,
)
t_load1 = perf_counter()
print(f"[Time] Data load & scenario build: {t_load1 - t_load0:.3f}s")


# ====== stage 1.5：FBBT / OBBT ======
# same as snog: FBBT open,OBBT open
obbt_solver_opts = {
    "NonConvex": 2,
    "MIPGap": 1,     
    "TimeLimit": 15   
}
for m, yvars in zip(model_list, first_stg_vars_list):
    tighten_bounds_one_model(m, yvars,
                             use_fbbt=False,
                             use_obbt=False,
                             obbt_solver_name="gurobi",
                             obbt_solver_opts=obbt_solver_opts,
                             max_rounds=3, tol=1e-6, verbose=True)

# ====== stage 2: Persistent Solver Packaging ======
ub_options = {
    'NonConvex': 2,        
}
lb_options = {
    'NonConvex': 2,
    'MIPGap': 1e-1,            
    'TimeLimit': 15    
}

t_wrap0 = perf_counter()
base_bundles = [BaseBundle(m, ub_options) for m in model_list]  
ms_bundles   = [MSBundle(m, yvars, lb_options) for m, yvars in zip(model_list, first_stg_vars_list)]  # LB 侧
t_wrap1 = perf_counter()
print(f"[Time] Persistent wrapper (GurobiPersistent) setup: {t_wrap1 - t_wrap0:.3f}s")

# ====== stage 3：main loop ======
agg_bundle = None  

t_run0 = perf_counter()
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=0,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle,
    gap_stop_tol=1e-4,
    plot_every=1,
    use_exact_opt=False,
    exact_solver_name="gurobi",
    exact_solver_opts={"NonConvex": 2, "TimeLimit": 60},
    time_limit=60*5*1
)
t_run1 = perf_counter()
print(f"[Time] Main loop total: {t_run1 - t_run0:.3f}s")

# ====== print output ======
print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")

[Time] Data load & scenario build: 0.112s
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2689754
Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter M

[Iter 0] evaluating Q for scenario 0
[Iter 0] evaluating Q for scenario 1
[Iter 0] evaluating Q for scenario 2
[Iter 0] evaluating Q for scenario 3
[BaseBundle.eval_at] Infeasible/Error for K=(-10.0, -19.32861065036913, 1000.0): status=warning, term=infeasibleOrUnbounded
[Iter 0] evaluating Q for scenario 4
[Iter 0] evaluating Q for scenario 5
[Iter 0] evaluating Q for scenario 6
[BaseBundle.eval_at] Infeasible/Error for K=(-10.0, -19.32861065036913, 1000.0): status=warning, term=infeasibleOrUnbounded
[Iter 0] evaluating Q for scenario 7
[Iter 0] evaluating Q for scenario 8
[BaseBundle.eval_at] Infeasible/Error for K=(-10.0, -19.32861065036913, 1000.0): status=warning, term=infeasibleOrUnbounded
[Iter 0] evaluating Q for scenario 9
[BaseBundle.eval_at] Infeasible/Error for K=(-10.0, -19.32861065036913, 1000.0): status=warning, term=infeasibleOrUnbounded

[Iter 0] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)      

[Iter 1] evaluating Q for scenario 0
[Iter 1] evaluating Q for scenario 1
[Iter 1] evaluating Q for scenario 2
[Iter 1] evaluating Q for scenario 3
[Iter 1] evaluating Q for scenario 4
[Iter 1] evaluating Q for scenario 5
[Iter 1] evaluating Q for scenario 6
[Iter 1] evaluating Q for scenario 7
[Iter 1] evaluating Q for scenario 8
[Iter 1] evaluating Q for scenario 9

[Iter 1] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T4        0       ms(dist)6.0172e+09     -6.0172e+09    2.2114e+02     2.2114e+02     (0.0000, -20.3434, 1000.0000) 

[Iter 1] subdivision type = face (code=3) on simplex T4
           face local verts = (3, 5, 1)
[Iter 1] Elapsed: 0.304s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 12-Core Proces

[Iter 2] evaluating Q for scenario 0
[BaseBundle.eval_at] Infeasible/Error for K=(7.927772342478549, -100.0, 13.972521163679772): status=warning, term=infeasibleOrUnbounded
[Iter 2] evaluating Q for scenario 1
[BaseBundle.eval_at] Infeasible/Error for K=(7.927772342478549, -100.0, 13.972521163679772): status=warning, term=infeasibleOrUnbounded
[Iter 2] evaluating Q for scenario 2
[BaseBundle.eval_at] Infeasible/Error for K=(7.927772342478549, -100.0, 13.972521163679772): status=warning, term=infeasibleOrUnbounded
[Iter 2] evaluating Q for scenario 3
[Iter 2] evaluating Q for scenario 4
[BaseBundle.eval_at] Infeasible/Error for K=(7.927772342478549, -100.0, 13.972521163679772): status=warning, term=infeasibleOrUnbounded
[Iter 2] evaluating Q for scenario 5
[BaseBundle.eval_at] Infeasible/Error for K=(7.927772342478549, -100.0, 13.972521163679772): status=warning, term=infeasibleOrUnbounded
[Iter 2] evaluating Q for scenario 6
[Iter 2] evaluating Q for scenario 7
[BaseBundle.eval_at] Inf

[Iter 3] evaluating Q for scenario 0
[Iter 3] evaluating Q for scenario 1
[Iter 3] evaluating Q for scenario 2
[Iter 3] evaluating Q for scenario 3
[Iter 3] evaluating Q for scenario 4
[Iter 3] evaluating Q for scenario 5
[Iter 3] evaluating Q for scenario 6
[Iter 3] evaluating Q for scenario 7
[Iter 3] evaluating Q for scenario 8
[Iter 3] evaluating Q for scenario 9

[Iter 3] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T7        3       ms(dist)6.0331e+09     -6.0331e+09    2.3618e+02     2.3626e+02     (2.1414, -21.4136, 1000.0000) 

[Iter 3] subdivision type = edge (code=2) on simplex T7
           edge local verts = (3, 5)
[Iter 3] Elapsed: 0.385s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 12-Core Processor

...

[Iter 27] evaluating Q for scenario 0
[Iter 27] evaluating Q for scenario 1
[Iter 27] evaluating Q for scenario 2
[Iter 27] evaluating Q for scenario 3
[Iter 27] evaluating Q for scenario 4
[Iter 27] evaluating Q for scenario 5
[Iter 27] evaluating Q for scenario 6
[Iter 27] evaluating Q for scenario 7
[Iter 27] evaluating Q for scenario 8
[Iter 27] evaluating Q for scenario 9

[Iter 27] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T61       5       ms(dist)9.8927e-01     -4.6019e-02    9.4325e-01     9.4325e-01     (9.9643, -99.6469, 0.4359)    

[Iter 27] subdivision type = edge (code=2) on simplex T61
           edge local verts = (31, 32)
[Iter 27] Elapsed: 0.685s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 1

[Iter 28] evaluating Q for scenario 0
[Iter 28] evaluating Q for scenario 1
[Iter 28] evaluating Q for scenario 2
[Iter 28] evaluating Q for scenario 3
[Iter 28] evaluating Q for scenario 4
[Iter 28] evaluating Q for scenario 5
[Iter 28] evaluating Q for scenario 6
[Iter 28] evaluating Q for scenario 7
[Iter 28] evaluating Q for scenario 8
[Iter 28] evaluating Q for scenario 9

[Iter 28] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T62       3       ms(dist)1.0975e+00     -2.6313e-02    1.0712e+00     1.0712e+00     (9.6696, -96.6979, -0.7668)   

[Iter 28] subdivision type = edge (code=2) on simplex T62
           edge local verts = (31, 33)
[Iter 28] Elapsed: 0.814s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 1

[Iter 29] evaluating Q for scenario 0
[Iter 29] evaluating Q for scenario 1
[Iter 29] evaluating Q for scenario 2
[Iter 29] evaluating Q for scenario 3
[Iter 29] evaluating Q for scenario 4
[Iter 29] evaluating Q for scenario 5
[Iter 29] evaluating Q for scenario 6
[Iter 29] evaluating Q for scenario 7
[Iter 29] evaluating Q for scenario 8
[Iter 29] evaluating Q for scenario 9

[Iter 29] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T57       4       ms(dist)1.4057e+00     -3.4917e-01    1.0566e+00     1.0566e+00     (9.3465, -93.4650, -8.4819)   

[Iter 29] subdivision type = edge (code=2) on simplex T57
           edge local verts = (4, 29)
[Iter 29] Elapsed: 0.707s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 12

[Iter 30] evaluating Q for scenario 0
[Iter 30] evaluating Q for scenario 1
[Iter 30] evaluating Q for scenario 2
[Iter 30] evaluating Q for scenario 3
[Iter 30] evaluating Q for scenario 4
[Iter 30] evaluating Q for scenario 5
[Iter 30] evaluating Q for scenario 6
[Iter 30] evaluating Q for scenario 7
[Iter 30] evaluating Q for scenario 8
[Iter 30] evaluating Q for scenario 9

[Iter 30] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T63       1       ms(dist)1.1232e+00     -1.7219e-01    9.5098e-01     9.5098e-01     (9.2702, -92.7016, 2.2099)    

[Iter 30] subdivision type = edge (code=2) on simplex T63
           edge local verts = (37, 29)
[Iter 30] Elapsed: 0.696s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 1

[Iter 31] evaluating Q for scenario 0
[Iter 31] evaluating Q for scenario 1
[Iter 31] evaluating Q for scenario 2
[Iter 31] evaluating Q for scenario 3
[Iter 31] evaluating Q for scenario 4
[Iter 31] evaluating Q for scenario 5
[Iter 31] evaluating Q for scenario 6
[Iter 31] evaluating Q for scenario 7
[Iter 31] evaluating Q for scenario 8
[Iter 31] evaluating Q for scenario 9

[Iter 31] Next Point Details:
Simplex   Scene   Type    As             ms             As+ms          Q              (Kp, Ki, Kd)                  
--------------------------------------------------------------------------------------------------------------------
T65       5       ms(dist)1.0815e+00     -1.3551e-01    9.4594e-01     9.4594e-01     (9.7790, -97.7896, 1.5704)    

[Iter 31] subdivision type = edge (code=2) on simplex T65
           edge local verts = (37, 30)
[Iter 31] Elapsed: 0.731s
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 7900X 1